# 06 — Classes, Objects & Traits

Notebooks 02 through 05 built a vocabulary for values, behaviour, and data. This notebook gives you the tools Scala uses to **package** them: classes for instances with state, objects for singletons, and traits for behaviour you can mix into multiple types.

If you have written Java, the shapes here will feel familiar — `class`, `extends`, methods, fields. The Scala twist is in the details: the primary constructor lives on the class line itself, singletons are a language-level concept, and traits can carry implementation in a way Java interfaces only recently learned to do.

## Defining a class

The class declaration in Scala 3 is unusually compact. The **primary constructor** is the class line itself — parameters in parentheses after the class name become the constructor parameters, and they are available throughout the body.

In [ ]:
class Point(x: Double, y: Double):
  def distanceToOrigin: Double = math.sqrt(x * x + y * y)

val p = Point(3.0, 4.0)
p.distanceToOrigin     // 5.0

A few things to notice in that small example:

- **No `new` keyword required.** `Point(3.0, 4.0)` is the constructor call. (You *can* still write `new Point(...)`; it works but is unnecessary in idiomatic Scala 3.)
- The parameters `x` and `y` are visible inside the body of the class but are **not public fields**. They are constructor parameters — accessible to methods of the class, invisible from outside.
- The body is a normal block expression. It can hold method definitions, value bindings, and side-effect statements that run at construction time.

## Making fields public

To turn a constructor parameter into a public field, prefix it with `val` (immutable) or `var` (mutable).

In [ ]:
class Point(val x: Double, val y: Double):
  def distanceToOrigin: Double = math.sqrt(x * x + y * y)

val p = Point(3.0, 4.0)
p.x                    // 3.0    — accessible from outside
p.distanceToOrigin     // 5.0

The pattern is precise:

| Prefix | Visibility outside | Reassignable inside |
|---|---|---|
| (none) | private to the class | no |
| `val` | public read-only | no |
| `var` | public read/write | yes |

Prefer `val`. Treat `var` fields like you treat `var` locals — only when there is a clear reason for mutable state.

## Methods and visibility

Method definitions inside a class look the same as the standalone methods from notebook 03. The difference is they can refer to the constructor parameters and to other members of the class via `this`.

In [ ]:
class BankAccount(val owner: String, private var balance: Double):

  def deposit(amount: Double): Unit =
    require(amount > 0, "deposit must be positive")
    balance += amount

  def withdraw(amount: Double): Unit =
    require(amount > 0 && amount <= balance, "invalid withdrawal")
    balance -= amount

  def currentBalance: Double = balance

val acc = BankAccount("alice", 100.0)
acc.deposit(50.0)
acc.withdraw(30.0)
acc.currentBalance     // 120.0
// acc.balance         // compile error: balance is private

Three visibility modifiers, in increasing strictness:

- **Public** (default) — accessible from anywhere.
- **`protected`** — accessible from this class and its subclasses.
- **`private`** — accessible only from inside this class.

There is also a `private[packagename]` form that scopes visibility to a package, which becomes useful in larger codebases.

## `object` — the singleton

An `object` defines a **single, named instance** that is created lazily the first time it's referenced. It has no constructor parameters. It is Scala's replacement for Java's `static` members and singletons rolled into one feature.

In [ ]:
object MathUtil:
  val pi: Double = 3.14159
  def circleArea(r: Double): Double = pi * r * r

MathUtil.pi               // 3.14159
MathUtil.circleArea(2.0)  // 12.56636

Use `object` for:

- **Utility functions** with no per-call state (`MathUtil.circleArea` above).
- **Application entry points** — paired with `@main` in notebook 01, but you can also write `object Main extends App`.
- **Constants** that don't belong to any specific instance.
- **Companion objects** — see the next section.

## Companion objects and `apply`

When an `object` is defined in the same file with the same name as a `class`, the two are **companions**. The companion object can see the class's private members, and vice versa. The companion is the natural place for factory methods, constants tied to the type, and the special method `apply`.

In [ ]:
class Temperature(val celsius: Double):
  def fahrenheit: Double = celsius * 9 / 5 + 32

object Temperature:
  val absoluteZero = Temperature(-273.15)

  def fromFahrenheit(f: Double): Temperature =
    Temperature((f - 32) * 5 / 9)

  def apply(c: Double): Temperature = new Temperature(c)

val room = Temperature(22.0)             // calls Temperature.apply(22.0)
val freezing = Temperature.fromFahrenheit(32)
Temperature.absoluteZero.celsius          // -273.15

The key trick is `apply`. When you write `Temperature(22.0)`, Scala interprets that as `Temperature.apply(22.0)` — calling the `apply` method on the companion object. You almost never need to write `apply` yourself for ordinary classes (Scala 3's auto-generated constructors handle the common case), but the same mechanism powers:

- `List(1, 2, 3)` — calling `List.apply(1, 2, 3)` on the `List` companion.
- `Map("a" -> 1)` — calling `Map.apply(...)`.
- Custom factories like `Temperature.fromFahrenheit` above, with a domain-specific name.

Read every `SomeName(...)` expression as *invoke `apply` on `SomeName`*. That single mental rule explains a lot of Scala's surface syntax.

## `trait` — composable behaviour

A **trait** is like an interface that can also carry implementation. Multiple traits can be mixed into a single class. A trait can declare abstract members, provide concrete methods, and hold its own state.

In [ ]:
trait Greeter:
  def name: String                          // abstract — must be supplied
  def greet: String = s"hello, $name"       // concrete — uses the abstract member

class Person(val name: String) extends Greeter

Person("ganesh").greet     // "hello, ganesh"

A class declares it implements a trait with `extends`. If a trait has abstract members (here, `name`), the class must provide them — or be declared `abstract` itself.

Traits cannot take constructor parameters in Scala 2; in Scala 3 they can. So this is also legal:

In [ ]:
trait Greeter(name: String):
  def greet: String = s"hello, $name"

class Person(name: String) extends Greeter(name)

## Mixin composition

The reason traits matter so much is that a class can extend **many** of them. This is called *mixin composition*. Use `extends` for the first one and `with` for each additional.

In [ ]:
trait HasName:
  def name: String

trait Greeter extends HasName:
  def greet: String = s"hello, $name"

trait Logger:
  def log(msg: String): Unit = println(s"[log] $msg")

class Person(val name: String) extends Greeter, Logger    // Scala 3 comma syntax
// (Scala 2: class Person(val name: String) extends Greeter with Logger)

val p = Person("ganesh")
p.greet                  // "hello, ganesh"
p.log("started")         // [log] started

Scala 3 lets you list multiple traits with **commas** after `extends`. The old `extends X with Y with Z` syntax still works.

Mixin composition is how you build small focused traits — `HasName`, `Greeter`, `Logger`, `Cacheable`, `Auditable` — and combine them only where needed. Each trait stays cohesive; the class declaration shows exactly which behaviours apply.

## Linearization (the brief version)

What happens when two traits define the same method? Scala uses **linearization** — it orders the trait hierarchy into a single line and resolves the method by walking that line from right to left.

For `class Person extends Greeter, Logger`, the linearization (simplified) is:

```
  Person -> Logger -> Greeter -> HasName -> Any
```

When you call a method, Scala starts from the leftmost type and looks rightward for the *first* concrete implementation. If a method is overridden in `Logger`, that override wins. If overridden in both, the rightmost mixin in the declaration wins — that's why the order in `extends A, B, C` matters.

Two practical takeaways:

- Most of the time you don't think about linearization. Conflicts are rare.
- When you do hit a conflict, `super[TraitName].method()` lets you call a specific ancestor's implementation explicitly.

We will revisit this in the type-system notebook (12) when it actually matters.

## `abstract class` vs `trait`

Both can carry implementation. Both can have abstract members. So when do you pick one over the other?

| Pick `abstract class` when | Pick `trait` when |
|---|---|
| You need a **single** primary constructor with parameters and you are on Scala 2 | You want **mixability** — multiple traits into one class |
| You are interoperating with Java code that expects a single base class | You are designing a behaviour that several unrelated types share |
| There is exactly one `is-a` relationship and you want to forbid `class C extends A, B` | The default for new Scala 3 code |

In practice, **default to `trait`** in Scala 3. Reach for `abstract class` only when one of the rare reasons applies.

## Extension methods (preview)

One last shape worth meeting now. Scala 3 lets you add methods to an existing type *from outside* using `extension`. The method becomes callable as if it were defined on the type itself.

In [ ]:
extension (s: String)
  def shout: String = s.toUpperCase + "!"

"hello".shout      // "HELLO!"

Extension methods are how you enrich types you don't own — `String`, `Int`, library classes — without subclassing them. Notebook 11 covers them in depth alongside `given` and `using`.

## Putting it together

A small model that uses all three pieces: a class with public fields, a trait with a default method, and a companion object with a factory and a constant.

In [ ]:
trait Identified:
  def id: String
  def describe: String = s"<id=$id>"

class User(val id: String, val email: String) extends Identified:
  override def describe: String = s"User($id, $email)"

object User:
  val anonymous = User("anon", "none@example.com")

  def fromEmail(email: String): User =
    User(email.takeWhile(_ != '@'), email)

val u = User.fromEmail("ganesh@example.com")
u.describe                  // User(ganesh, ganesh@example.com)
User.anonymous.describe     // User(anon, none@example.com)

Notice four things:

1. `User` extends `Identified` and supplies the abstract `id` via the constructor parameter.
2. `User.describe` **overrides** the trait's default. The `override` keyword is required.
3. The companion object exposes `anonymous` (a constant) and `fromEmail` (a factory) — both natural fits for a companion.
4. The class and the companion share visibility — the companion can see the class's private members, which is occasionally useful for things like `equals` and `hashCode`.

## What's next

You now have the imperative-OOP shape of Scala. Notebook 07 introduces the form that most real Scala code reaches for: **case classes** (immutable data records with built-in `equals`, `hashCode`, `toString`, and `copy`) and **enums** (Scala 3's tidy way to express finite sum types). Together they form the algebraic-data-type backbone that pattern matching in notebook 08 will exploit.